# UN MUERTO MAS

In [10]:
import requests
from urllib.parse import quote

HOST = "https://api.chartmetric.com"
REFRESH_TOKEN = "112E1dROGObsUx9hWKnrOxLQWrjDZ9YXwoByI9ynpN7QjtFY2Lq1E3uW7zvuIrpY"

ARTIST_NAME = "Adrián Berra"

# Obtener access token
res = requests.post(
    f"{HOST}/api/token",
    json={"refreshtoken": REFRESH_TOKEN}
)
res.raise_for_status()

access_token = res.json()["token"]

headers = {
    "Authorization": f"Bearer {access_token}",
    "X-Accept-Partial-Data": "true"
}

# Buscar artista por nombre
query = quote(ARTIST_NAME)
search_url = f"{HOST}/api/search?q={query}&type=artists&limit=10"

res = requests.get(search_url, headers=headers)
print("SEARCH STATUS:", res.status_code)
res.raise_for_status()

search_data = res.json()

print("\nRespuesta completa de search:")
print(search_data)

# Extraer resultados de artistas
artists = search_data.get("obj", {}).get("artists", [])

print("\nResultados encontrados:")
for i, artist in enumerate(artists):
    print(f"\nResultado {i}")
    print(artist)

# Tomar el primer resultado
if len(artists) == 0:
    print("\nNo se encontró ningún artista con ese nombre.")
else:
    artist_id = artists[0].get("id") or artists[0].get("cm_artist")
    artist_name = artists[0].get("name")

    print("\nChartmetric ID elegido:", artist_id)
    print("Nombre elegido:", artist_name)

    # Pedir metadata
    metadata_url = f"{HOST}/api/artist/{artist_id}"
    res_meta = requests.get(metadata_url, headers=headers)

    print("\nMETADATA STATUS:", res_meta.status_code)
    res_meta.raise_for_status()

    metadata = res_meta.json()

    print("\nMetadata:")
    print(metadata)

SEARCH STATUS: 200

Respuesta completa de search:
{'obj': {'artists': [{'id': 316653, 'name': 'Adrián Berra', 'image_url': 'https://i.scdn.co/image/ab6761610000e5eb1f5a6c0b799e76476a96d8a8', 'isni': None, 'verified': False, 'sp_followers': 75469, 'sp_monthly_listeners': 321324, 'cm_artist_score': 59.34138843594469, 'band': False, 'gender': 'male', 'primary_genre_smart': 501282}, {'id': 3767998, 'name': 'Adrian Berra', 'image_url': None, 'isni': None, 'verified': True, 'sp_followers': 715, 'sp_monthly_listeners': 3, 'cm_artist_score': 0.5481864225586053, 'band': None, 'gender': None, 'primary_genre_smart': 501282}, {'id': 4521350, 'name': 'Adrian Berra y la Vaca Perdida', 'image_url': None, 'isni': None, 'verified': True, 'sp_followers': 171, 'sp_monthly_listeners': 7, 'cm_artist_score': 0.01566707678939405, 'band': None, 'gender': None, 'primary_genre_smart': 501120}, {'id': 14790375, 'name': 'Adrián Berra', 'image_url': None, 'isni': None, 'verified': True, 'sp_followers': None, 'sp_m

In [11]:
import pandas as pd
import json

# ============================================================
# Exportar metadata de un artista a Excel en formato campo-valor
# ============================================================

artist_obj = metadata["obj"]

def flatten_dict(d, parent_key="", sep="."):
    items = []

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k

        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())

        elif isinstance(v, list):
            if all(isinstance(x, dict) for x in v):
                items.append((new_key, json.dumps(v, ensure_ascii=False)))
            else:
                items.append((new_key, ", ".join(map(str, v))))

        else:
            items.append((new_key, v))

    return dict(items)

metadata_flat = flatten_dict(artist_obj)

df_metadata_excel = pd.DataFrame(
    list(metadata_flat.items()),
    columns=["campo", "valor"]
)

# Reordenar para que el id quede primero
df_metadata_excel["orden"] = df_metadata_excel["campo"].apply(
    lambda x: 0 if x == "id" else 1
)

df_metadata_excel = (
    df_metadata_excel
    .sort_values(["orden", "campo"])
    .drop(columns="orden")
    .reset_index(drop=True)
)

nombre_archivo = "metadata_adrian_berra.xlsx"

df_metadata_excel.to_excel(nombre_archivo, index=False)

print("Archivo exportado:", nombre_archivo)
display(df_metadata_excel.head(30))

Archivo exportado: metadata_adrian_berra.xlsx


,campo,valor
0,id,316653
1,activities,"[{""id"": 504880, ""name"": ""vacation""}, {""id"": 50..."
2,band,False
3,band_members,None
4,booking_agent,None
5,career_status.stage,developing
6,career_status.stage_score,86
7,career_status.trend,steady
8,career_status.trend_score,79
9,cm_artist_rank,83913


In [12]:
import requests
import pandas as pd
import json
from datetime import date, timedelta

HOST = "https://api.chartmetric.com"
ARTIST_ID = 316653

# Reutilizar access_token si ya existe en memoria
headers = {
    "Authorization": f"Bearer {access_token}",
    "X-Accept-Partial-Data": "true"
}

since = "2023-04-27"
until = "2026-04-27"

combinaciones = {
    "instagram": {
        "followers": ["country", "city", "interest", "brand", "language", "stat", "demographic"],
        "likes": ["country", "city", "interest", "brand", "language", "stat", "demographic"]
    },
    "youtube": {
        "followers": ["country", "language", "stat", "demographic"],
        "commenters": ["country", "language", "stat", "demographic"]
    },
    "tiktok": {
        "followers": ["country", "language", "stat", "demographic"]
    }
}

def normalizar_respuesta(obj, domain, audience_type, stats_type, status_code, url):
    filas = []

    if isinstance(obj, list):
        for item in obj:
            fila = {
                "artist_id": ARTIST_ID,
                "domain": domain,
                "audience_type": audience_type,
                "stats_type": stats_type,
                "status_code": status_code,
                "url": url
            }
            if isinstance(item, dict):
                fila.update(item)
            else:
                fila["value"] = item
            filas.append(fila)

    elif isinstance(obj, dict):
        fila = {
            "artist_id": ARTIST_ID,
            "domain": domain,
            "audience_type": audience_type,
            "stats_type": stats_type,
            "status_code": status_code,
            "url": url
        }
        fila.update(obj)
        filas.append(fila)

    else:
        filas.append({
            "artist_id": ARTIST_ID,
            "domain": domain,
            "audience_type": audience_type,
            "stats_type": stats_type,
            "status_code": status_code,
            "url": url,
            "value": obj
        })

    return filas

todas_las_filas = []
resumen_requests = []

for domain, audience_dict in combinaciones.items():
    for audience_type, stats_types in audience_dict.items():
        for stats_type in stats_types:

            url = (
                f"{HOST}/api/artist/{ARTIST_ID}/social-audience-stats"
                f"?domain={domain}"
                f"&audienceType={audience_type}"
                f"&statsType={stats_type}"
                f"&since={since}"
                f"&until={until}"
                f"&limit=100"
                f"&offset=0"
            )

            res = requests.get(url, headers=headers)
            status_code = res.status_code

            resumen_requests.append({
                "artist_id": ARTIST_ID,
                "domain": domain,
                "audience_type": audience_type,
                "stats_type": stats_type,
                "status_code": status_code,
                "url": url
            })

            print(domain, audience_type, stats_type, "status:", status_code)

            if status_code == 200:
                data = res.json()
                obj = data.get("obj", [])

                filas = normalizar_respuesta(
                    obj=obj,
                    domain=domain,
                    audience_type=audience_type,
                    stats_type=stats_type,
                    status_code=status_code,
                    url=url
                )

                todas_las_filas.extend(filas)
            else:
                todas_las_filas.append({
                    "artist_id": ARTIST_ID,
                    "domain": domain,
                    "audience_type": audience_type,
                    "stats_type": stats_type,
                    "status_code": status_code,
                    "url": url,
                    "error_text": res.text
                })

df_audience = pd.DataFrame(todas_las_filas)
df_resumen = pd.DataFrame(resumen_requests)



instagram followers country status: 200
instagram followers city status: 200
instagram followers interest status: 200
instagram followers brand status: 200
instagram followers language status: 200
instagram followers stat status: 200
instagram followers demographic status: 200
instagram likes country status: 200
instagram likes city status: 200
instagram likes interest status: 200
instagram likes brand status: 200
instagram likes language status: 200
instagram likes stat status: 200
instagram likes demographic status: 200
youtube followers country status: 200
youtube followers language status: 200
youtube followers stat status: 200
youtube followers demographic status: 200
youtube commenters country status: 200
youtube commenters language status: 200
youtube commenters stat status: 200
youtube commenters demographic status: 200
tiktok followers country status: 200
tiktok followers language status: 200
tiktok followers stat status: 200
tiktok followers demographic status: 200


In [13]:

df_audience.to_csv(
    "social_audience_stats_adrian_berra_3_anios.csv",
    index=False,
    encoding="utf-8-sig"
)

df_resumen.to_csv(
    "social_audience_stats_adrian_berra_3_anios_resumen.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV exportados")

CSV exportados


In [15]:
import requests
import pandas as pd

# ============================================================
# CONFIGURACIÓN
# ============================================================

HOST = "https://api.chartmetric.com"
ARTIST_ID = 316653   # un muerto mas

headers = {
    "Authorization": f"Bearer {access_token}",
    "X-Accept-Partial-Data": "true"
}

since = "2023-04-27"
until = "2026-04-27"

# ============================================================
# REQUEST
# ============================================================

url = (
    f"{HOST}/api/artist/{ARTIST_ID}/where-people-listen"
    f"?since={since}"
    f"&until={until}"
    f"&limit=50"
    f"&offset=0"
    f"&includeEstimates=true"
)

res = requests.get(url, headers=headers)

print("STATUS:", res.status_code)
print("URL:", url)

res.raise_for_status()

data = res.json()
obj = data.get("obj", {})

# ============================================================
# PARSE CITIES
# ============================================================

filas_cities = []

cities = obj.get("cities", {})

for city_name, registros in cities.items():
    for r in registros:
        fila = {
            "artist_id": ARTIST_ID,
            "location_type": "city",
            "location_name": city_name
        }
        fila.update(r)
        filas_cities.append(fila)

df_cities = pd.DataFrame(filas_cities)

# ============================================================
# PARSE COUNTRIES
# ============================================================

filas_countries = []

countries = obj.get("countries", {})

for country_name, registros in countries.items():
    for r in registros:
        fila = {
            "artist_id": ARTIST_ID,
            "location_type": "country",
            "location_name": country_name
        }
        fila.update(r)
        filas_countries.append(fila)

df_countries = pd.DataFrame(filas_countries)



STATUS: 200
URL: https://api.chartmetric.com/api/artist/316653/where-people-listen?since=2023-04-27&until=2026-04-27&limit=50&offset=0&includeEstimates=true


In [16]:


# ============================================================
# EXPORTAR CSV
# ============================================================

df_cities.to_csv(
    "where_people_listen_adrian_berra_3_anios_cities.csv",
    index=False,
    encoding="utf-8-sig"
)

df_countries.to_csv(
    "where_people_listen_adrian_berra_3_anios_countries.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV exportados:")

print("Filas cities:", len(df_cities))
print("Filas countries:", len(df_countries))

CSV exportados:
Filas cities: 13201
Filas countries: 8276


# UN MUERTO MAS FIN

In [ ]:
import requests
import time
import json
from pathlib import Path

REFRESH_TOKEN = "MI TOKEN"
ARTIST_ID = 2762

outdir = Path("chartmetric_test")
outdir.mkdir(exist_ok=True)

# 1) token
token_resp = requests.post(
    "https://api.chartmetric.com/api/token",
    json={"refreshtoken": REFRESH_TOKEN}
)
token_resp.raise_for_status()
access_token = token_resp.json()["token"]

headers = {
    "Authorization": f"Bearer {access_token}",
    "X-Accept-Partial-Data": "true"
}

time.sleep(1.1)

# 2) metadata
meta_resp = requests.get(
    f"https://api.chartmetric.com/api/artist/{ARTIST_ID}",
    headers=headers
)
meta_resp.raise_for_status()
meta_json = meta_resp.json()

with open(outdir / f"artist_{ARTIST_ID}_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta_json, f, ensure_ascii=False, indent=2)

time.sleep(1.1)

# 3) past events
events_resp = requests.get(
    f"https://api.chartmetric.com/api/artist/{ARTIST_ID}/past/events",
    headers=headers,
    params={"fromDaysAgo": 365, "toDaysAgo": 0, "limit": 5, "offset": 0}
)
events_resp.raise_for_status()
events_json = events_resp.json()

with open(outdir / f"artist_{ARTIST_ID}_past_events.json", "w", encoding="utf-8") as f:
    json.dump(events_json, f, ensure_ascii=False, indent=2)

print("Archivos guardados:")
print(outdir / f"artist_{ARTIST_ID}_metadata.json")
print(outdir / f"artist_{ARTIST_ID}_past_events.json")

Archivos guardados:
chartmetric_test\artist_2762_metadata.json
chartmetric_test\artist_2762_past_events.json


In [2]:
import json
from pprint import pprint

with open("chartmetric_test/artist_2762_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open("chartmetric_test/artist_2762_past_events.json", "r", encoding="utf-8") as f:
    past_events = json.load(f)

print("=== KEYS METADATA ===")
print(metadata.keys())

print("\n=== KEYS PAST EVENTS ===")
print(past_events.keys())

=== KEYS METADATA ===
dict_keys(['obj'])

=== KEYS PAST EVENTS ===
dict_keys(['obj'])


In [3]:
print("=== TIPO metadata['obj'] ===")
print(type(metadata["obj"]))

print("\n=== TIPO past_events['obj'] ===")
print(type(past_events["obj"]))

print("\n=== PRIMERAS CLAVES DEL OBJETO DE METADATA ===")
print(metadata["obj"].keys())

print("\n=== CANTIDAD DE EVENTOS ===")
print(len(past_events["obj"]))

=== TIPO metadata['obj'] ===
<class 'dict'>

=== TIPO past_events['obj'] ===
<class 'list'>

=== PRIMERAS CLAVES DEL OBJETO DE METADATA ===
dict_keys(['id', 'name', 'created_at', 'code2', 'band', 'pronoun_title', 'gender_title', 'gender', 'isni', 'cm_artist_rank', 'cm_artist_score', 'cover_url', 'image_url', 'hometown_city', 'current_city', 'current_city_id', 'band_members', 'booking_agent', 'record_label', 'press_contact', 'general_manager', 'topSongwriterCollaborators', 'description', 'genres', 'genre_smart_ordered', 'moods', 'activities', 'career_status', 'cm_statistics', 'genreRank', 'subGenreRank1', 'subGenreRank2'])

=== CANTIDAD DE EVENTOS ===
0


In [4]:
print("=== METADATA COMPLETA DEL ARTISTA ===")
pprint(metadata["obj"])

=== METADATA COMPLETA DEL ARTISTA ===
{'activities': [{'id': 504873, 'name': 'daydreaming'},
                {'id': 504872, 'name': 'dancy'},
                {'id': 504877, 'name': 'self-love'},
                {'id': 504871, 'name': 'bonding'},
                {'id': 504887, 'name': 'reassured'},
                {'id': 504876, 'name': 'partying'},
                {'id': 504879, 'name': 'rural'},
                {'id': 504874, 'name': 'driving'}],
 'band': False,
 'band_members': None,
 'booking_agent': None,
 'career_status': {'stage': 'superstar',
                   'stage_score': 89,
                   'trend': 'growth',
                   'trend_score': 69},
 'cm_artist_rank': 1,
 'cm_artist_score': 99.38711679376492,
 'cm_statistics': {'boomplay_comments': None,
                   'boomplay_comments_rank': None,
                   'boomplay_favorites': None,
                   'boomplay_favorites_rank': None,
                   'boomplay_plays': None,
                   'boomplay_

In [6]:
print("=== PRIMER EVENTO ===")
pprint(past_events["obj"])

=== PRIMER EVENTO ===
[]


In [14]:
time.sleep(1.1)

events_resp = requests.get(
    f"https://api.chartmetric.com/api/artist/{ARTIST_ID}/past/events",
    headers=headers,
    params={
        "fromDaysAgo": 60,
        "toDaysAgo": 0,
        "limit": 10,
        "offset": 0
    }
)

print(events_resp.status_code)
print(events_resp.url)

events_resp.raise_for_status()
events_json = events_resp.json()

with open(outdir / f"artist_{ARTIST_ID}_past_events_doc_example.json", "w", encoding="utf-8") as f:
    json.dump(events_json, f, ensure_ascii=False, indent=2)

print("Cantidad de eventos:", len(events_json.get("obj", [])))

200
https://api.chartmetric.com/api/artist/2762/past/events?fromDaysAgo=60&toDaysAgo=0&limit=10&offset=0
Cantidad de eventos: 0


In [16]:
# 4) current events - con parámetros de tiempo
import time
import json
from pprint import pprint

time.sleep(1.1)

current_resp = requests.get(
    f"https://api.chartmetric.com/api/artist/{ARTIST_ID}/current/events",
    headers=headers,
    params={
        "fromDaysAgo": 0,
        "toDaysAgo": -50,
        "limit": 10,
        "offset": 0
    }
)

print("=== STATUS CURRENT ===")
print(current_resp.status_code)

print("\n=== URL CURRENT ===")
print(current_resp.url)

# si falla, mostremos el texto de error antes de cortar
if current_resp.status_code != 200:
    print("\n=== ERROR TEXT ===")
    print(current_resp.text)

current_resp.raise_for_status()
current_json = current_resp.json()

with open(outdir / f"artist_{ARTIST_ID}_current_events.json", "w", encoding="utf-8") as f:
    json.dump(current_json, f, ensure_ascii=False, indent=2)

print("\n=== KEYS DEL JSON CURRENT ===")
print(current_json.keys())

print("\n=== CANTIDAD DE CURRENT EVENTS ===")
print(len(current_json.get("obj", [])))

if len(current_json.get("obj", [])) > 0:
    print("\n=== PRIMER CURRENT EVENT ===")
    pprint(current_json["obj"][0])
else:
    print("\nNo devolvió current events.")

=== STATUS CURRENT ===
200

=== URL CURRENT ===
https://api.chartmetric.com/api/artist/2762/current/events?fromDaysAgo=0&toDaysAgo=-50&limit=10&offset=0

=== KEYS DEL JSON CURRENT ===
dict_keys(['obj'])

=== CANTIDAD DE CURRENT EVENTS ===
0

No devolvió current events.


In [17]:
import requests
import time
import json
from pathlib import Path
from pprint import pprint

# carpeta de salida
outdir = Path("chartmetric_test_multi")
outdir.mkdir(exist_ok=True)

# artistas de prueba tomados de tu head
artists = [
    {"id": 5381, "name": "Dua Lipa"},
    {"id": 214945, "name": "Bad Bunny"},
]

# helper para nombre de archivo seguro
def safe_name(name):
    return (
        name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
    )

results_summary = []

for artist in artists:
    artist_id = artist["id"]
    artist_name = artist["name"]
    slug = safe_name(artist_name)

    print("\n" + "=" * 70)
    print(f"ARTISTA: {artist_name} ({artist_id})")
    print("=" * 70)

    # ---------- METADATA ----------
    time.sleep(1.1)

    meta_resp = requests.get(
        f"https://api.chartmetric.com/api/artist/{artist_id}",
        headers=headers
    )

    print("\n[METADATA]")
    print("status:", meta_resp.status_code)
    print("url:", meta_resp.url)

    if meta_resp.status_code != 200:
        print("error metadata:", meta_resp.text)
        results_summary.append({
            "artist_id": artist_id,
            "artist_name": artist_name,
            "metadata_status": meta_resp.status_code,
            "past_events_status": None,
            "current_events_status": None,
            "n_past_events": None,
            "n_current_events": None,
        })
        continue

    meta_json = meta_resp.json()

    with open(outdir / f"artist_{artist_id}_{slug}_metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta_json, f, ensure_ascii=False, indent=2)

    meta_obj = meta_json.get("obj", {})
    cm_stats = meta_obj.get("cm_statistics", {})

    print("name:", meta_obj.get("name"))
    print("sp_monthly_listeners:", cm_stats.get("sp_monthly_listeners"))
    print("sp_followers:", cm_stats.get("sp_followers"))
    print("ins_followers:", cm_stats.get("ins_followers"))
    print("tiktok_followers:", cm_stats.get("tiktok_followers"))
    print("ycs_subscribers:", cm_stats.get("ycs_subscribers"))

    # ---------- PAST EVENTS ----------
    time.sleep(1.1)

    past_resp = requests.get(
        f"https://api.chartmetric.com/api/artist/{artist_id}/past/events",
        headers=headers,
        params={
            "fromDaysAgo": 365,
            "toDaysAgo": 0,
            "limit": 10,
            "offset": 0
        }
    )

    print("\n[PAST EVENTS]")
    print("status:", past_resp.status_code)
    print("url:", past_resp.url)

    if past_resp.status_code != 200:
        print("error past:", past_resp.text)
        n_past = None
    else:
        past_json = past_resp.json()
        with open(outdir / f"artist_{artist_id}_{slug}_past_events.json", "w", encoding="utf-8") as f:
            json.dump(past_json, f, ensure_ascii=False, indent=2)

        n_past = len(past_json.get("obj", []))
        print("cantidad past events:", n_past)

        if n_past > 0:
            print("keys primer past event:")
            print(past_json["obj"][0].keys())

    # ---------- CURRENT EVENTS ----------
    time.sleep(1.1)

    current_resp = requests.get(
        f"https://api.chartmetric.com/api/artist/{artist_id}/current/events",
        headers=headers,
        params={
            "fromDaysAgo": 0,
            "toDaysAgo": -50,
            "limit": 10,
            "offset": 0
        }
    )

    print("\n[CURRENT EVENTS]")
    print("status:", current_resp.status_code)
    print("url:", current_resp.url)

    if current_resp.status_code != 200:
        print("error current:", current_resp.text)
        n_current = None
    else:
        current_json = current_resp.json()
        with open(outdir / f"artist_{artist_id}_{slug}_current_events.json", "w", encoding="utf-8") as f:
            json.dump(current_json, f, ensure_ascii=False, indent=2)

        n_current = len(current_json.get("obj", []))
        print("cantidad current events:", n_current)

        if n_current > 0:
            print("keys primer current event:")
            print(current_json["obj"][0].keys())

    results_summary.append({
        "artist_id": artist_id,
        "artist_name": artist_name,
        "metadata_status": meta_resp.status_code,
        "past_events_status": past_resp.status_code,
        "current_events_status": current_resp.status_code,
        "n_past_events": n_past,
        "n_current_events": n_current,
    })

print("\n" + "=" * 70)
print("RESUMEN FINAL")
print("=" * 70)
pprint(results_summary)


ARTISTA: Dua Lipa (5381)

[METADATA]
status: 200
url: https://api.chartmetric.com/api/artist/5381
name: Dua Lipa
sp_monthly_listeners: 65015229
sp_followers: 47767361
ins_followers: 88665348
tiktok_followers: 11200000
ycs_subscribers: 24700000

[PAST EVENTS]
status: 200
url: https://api.chartmetric.com/api/artist/5381/past/events?fromDaysAgo=365&toDaysAgo=0&limit=10&offset=0
cantidad past events: 10
keys primer past event:
dict_keys(['id', 'event_name', 'jambase_event_url', 'songkick_event_url', 'seatgeek_event_url', 'ticketmaster_event_url', 'image_url', 'type', 'start_date', 'end_date', 'venue_id', 'venue_name', 'venue_url', 'venue_capacity', 'city', 'code2', 'low_price', 'high_price', 'price', 'price_trend', 'currency', 'is_headliner'])

[CURRENT EVENTS]
status: 200
url: https://api.chartmetric.com/api/artist/5381/current/events?fromDaysAgo=0&toDaysAgo=-50&limit=10&offset=0
cantidad current events: 0

ARTISTA: Bad Bunny (214945)

[METADATA]
status: 200
url: https://api.chartmetric.